# ETL walkthrough: build the LabelSage catalog and look at the data

Runs the same phases as `run.py build`, one at a time, and shows what each phase did. The code lives in the
`.py` modules; this notebook only calls them and looks at the results.

- **Sources:** FDC (USDA) is the main source for identity, ingredients and nutrients. Open Food Facts (OFF) fills
  gaps and gives the photo, only when its record passes the identity check against FDC.
- **Scenarios:** set `SCENARIO` and `KEEP_OFF_ONLY` below. Each scenario writes to its own folder in
  `~/Projects/ai-shopping-agent-data/out/notebook/<scenario>/`.
- **Before committing:** clear all outputs (Edit > Clear All Outputs), so data stays out of git.

Sections 1 to 8 build the data. Sections 9 to 12 look at the results: the identity-check model (9), the phone
catalog (10), the server export (11) and a one-table summary (12).

Run top to bottom. A full run takes about 5 to 7 minutes.

## 0. Setup

In [ ]:
import pathlib, sys, time, json
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ETL = pathlib.Path.cwd()                      # open the notebook from app/backend/etl
sys.path.insert(0, str(ETL))
from extract import extract
from transform import transform
from load import load
from match import matcher, evaluate

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# ---- scenario settings ----
SCENARIO = "fdc-first"                        # name of the output folder
transform.KEEP_OFF_ONLY = False               # True = also keep products only Open Food Facts has
OUT = load.OUT_DIR / "notebook" / SCENARIO

extract.WORK_DIR.mkdir(parents=True, exist_ok=True)
work_db = extract.WORK_DIR / f"notebook-{SCENARIO}.duckdb"
work_db.unlink(missing_ok=True)
con = duckdb.connect(str(work_db))
con.execute("SET enable_progress_bar = false")
q = lambda sql: con.sql(sql).df()             # run SQL, get a DataFrame
print("scenario:", SCENARIO, "| keep OFF-only:", transform.KEEP_OFF_ONLY, "| output:", OUT)

## 1. Extract: which source files this build uses
Downloads happen with `python3 run.py download`; this only lists the newest copy of each source.

In [ ]:
pd.DataFrame([{"source": s, "file": (f := extract.latest_file(s)).name, "downloaded": f.parent.name,
               "MB": round(f.stat().st_size / 1e6, 1)} for s in extract.SOURCES])

## 2. Transform phase 1: load the sources into DuckDB

In [ ]:
t = time.time()
transform.load_sources(con)
print(f"{time.time() - t:.0f}s")
q("SELECT count(*) AS fdc_branded_rows FROM fdc_branded_food")

## 3. Transform phase 2: barcodes
Every barcode becomes a GTIN-14 with a valid check digit; broken barcodes are dropped; the newest record per
barcode is kept. FDC keeps US-market products only.

In [ ]:
t = time.time()
stats = transform.key_and_dedup(con)
print(f"{time.time() - t:.0f}s", stats)

In [ ]:
# what the barcode rules removed (the OFF count scans the whole gzip file once, about a minute)
raw = q("""
    SELECT 'FDC' AS source, count(*) AS raw_rows,
           count(*) FILTER (WHERE NOT gtin_ok(gtin14(gtin_upc))) AS bad_barcode
    FROM fdc_branded_food
    UNION ALL
    SELECT 'OFF (US + ID)', count(*), count(*) FILTER (WHERE NOT gtin_ok(gtin14(code))) FROM off_raw""")
raw["kept"] = [stats["fdc_products"], stats["off_products_us_and_id"]]
raw["duplicates_or_other"] = raw.raw_rows - raw.bad_barcode - raw.kept
raw

In [ ]:
# OFF "US" products by GS1 country prefix: prefixes 000-139 are US/Canada; others are foreign products
# that someone tagged as sold in the US (611 = Morocco was the top-scanned in the 2026-10-27 build)
q("""
    SELECT substr(g, 2, 3) AS gs1_prefix, count(*) AS products, sum(coalesce(unique_scans, 0)) AS scans,
           CAST(substr(g, 2, 3) AS INTEGER) BETWEEN 0 AND 139 AS us_or_canada
    FROM off WHERE region LIKE '%us%'
    GROUP BY ALL ORDER BY scans DESC LIMIT 15""")

## 4. Overlap: which products are in which source
The FDC-first rule keeps FDC products. With `KEEP_OFF_ONLY = False`, the OFF-only rows below are left out:
this table shows what that costs.

In [ ]:
overlap = q("""
    SELECT CASE WHEN f.g IS NOT NULL AND o.g IS NOT NULL THEN 'both'
                WHEN f.g IS NOT NULL THEN 'FDC only' ELSE 'OFF only' END AS in_source,
           count(*) AS products,
           count(*) FILTER (WHERE coalesce(f.ingredients, o.ingredients) IS NOT NULL) AS with_ingredients,
           count(*) FILTER (WHERE o.image_url IS NOT NULL) AS with_off_image,
           count(*) FILTER (WHERE o.region LIKE '%id%') AS indonesian,
           sum(coalesce(o.unique_scans, 0)) AS off_scans
    FROM fdc f FULL OUTER JOIN off o ON o.g = f.g
    GROUP BY 1 ORDER BY products DESC""")
overlap

In [ ]:
overlap.set_index("in_source")[["products", "with_ingredients", "with_off_image"]].plot.bar(figsize=(8, 3.5), rot=0)
plt.title("Products per source"); plt.ylabel("products"); plt.tight_layout(); plt.show()

## 5. Transform phase 3: combine, identity check, cleanup
FDC wins name, brand, category, ingredients and nutrients. Each OFF record that shares a barcode with FDC is
scored against it (`match/matcher.py`, method `same_barcode`); only `accept` keeps the OFF photo.

In [ ]:
t = time.time()
transform.combine_and_clean(con)
print(f"{time.time() - t:.0f}s")
q("SELECT count(*) AS products FROM merged")

### 5a. Identity check: OFF record vs FDC on the same barcode

In [ ]:
q("""SELECT tier, count(*) AS pairs, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
             round(avg(confidence), 3) AS avg_confidence
      FROM off_check GROUP BY tier ORDER BY pairs DESC""")

In [ ]:
conf = q("SELECT confidence FROM off_check")
conf.confidence.plot.hist(bins=50, figsize=(8, 3.5))
for x in matcher.THRESHOLDS["same_barcode"]:
    plt.axvline(x, color="black", linestyle="--")
plt.title("Identity check confidence (dashed: review and accept thresholds)"); plt.xlabel("confidence")
plt.tight_layout(); plt.show()

In [ ]:
# why pairs were not accepted: the most common reasons
q("""SELECT tier, coalesce(nullif(split_part(r, ':', 1), ''), '(low similarity only)') AS reason, count(*) AS pairs
      FROM (SELECT tier, unnest(string_split(coalesce(reasons, ''), ',')) AS r FROM off_check WHERE tier <> 'accept')
      GROUP BY ALL ORDER BY pairs DESC LIMIT 12""")

In [ ]:
# examples per tier, to judge by eye (FDC is the reference, OFF the record being checked)
examples = q("""
    SELECT c.tier, c.confidence, f.name AS fdc_name, f.brand AS fdc_brand, o.name AS off_name, o.brand AS off_brand,
           c.reasons
    FROM off_check c JOIN fdc f ON CAST(f.g AS BIGINT) = c.gtin JOIN off o ON o.g = f.g
    USING SAMPLE 3000 ROWS (reservoir, 7)""")
examples.sort_values("confidence").groupby("tier").head(8)

### 5b. A labelling sample, to measure precision
Writes about 300 pairs to `OUT/label-sample.csv`, spread over the confidence range. Fill the `same_product`
column with 1 or 0; section 9 then measures precision and recall on real pairs. Target: no wrong photo among the
accepted pairs. The cell skips writing when the file already exists, so a rerun never erases labels.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / "label-sample.csv").exists():
    print("label-sample.csv already exists: kept (delete it to draw a new sample)")
else:
  sample = q("""
    SELECT lpad(CAST(c.gtin AS VARCHAR), 14, '0') AS gtin, c.tier, c.confidence, c.reasons,
           f.name AS fdc_name, f.brand AS fdc_brand, f.brand_owner AS fdc_brand_owner,
           o.name AS off_name, o.brand AS off_brand, o.image_url AS off_image, NULL AS same_product
    FROM off_check c JOIN fdc f ON CAST(f.g AS BIGINT) = c.gtin JOIN off o ON o.g = f.g
    QUALIFY row_number() OVER (PARTITION BY round(c.confidence * 20) ORDER BY hash(c.gtin)) <= 15""")
  sample.to_csv(OUT / "label-sample.csv", index=False)
  print(len(sample), "pairs ->", OUT / "label-sample.csv")

### 5c. Field completeness after the merge, and where each field came from

In [ ]:
fields = ["name", "brand", "brand_owner", "category", "ingredients", "serving_size", "size", "nova", "image_url",
          "kcal", "protein_g", "sugar_g", "sodium_mg"]
total = q("SELECT count(*) AS n FROM merged").n[0]
pd.DataFrame([{"field": f, "filled": (n := q(f"SELECT count({f}) AS n FROM merged").n[0]),
               "pct": round(100 * n / total, 1)} for f in fields]).set_index("field")

In [ ]:
q("""SELECT CASE WHEN ingredients IS NULL THEN 'none' WHEN ingredients_from_off THEN 'OFF (FDC had none)'
                  ELSE 'FDC' END AS ingredients_from,
             count(*) AS products,
             count(*) FILTER (WHERE recipe_changed) AS recipe_may_have_changed
      FROM merged GROUP BY 1 ORDER BY products DESC""")

In [ ]:
# nutrients that had to come from OFF because FDC had none (bit per nutrient)
pd.DataFrame([{"nutrient": n, "from_off": q(f"SELECT count(*) AS c FROM merged WHERE from_off_mask & {1 << i} > 0").c[0]}
              for i, n in enumerate(transform.NUTRIENTS)]).set_index("nutrient").T

In [ ]:
# ingredient lists: language and quality gate (failed lists are hidden on the phone: "Can't confirm")
q("""SELECT ingredients_lang AS language, count(*) AS lists, count(*) FILTER (WHERE NOT ingredients_ok) AS failed_gate
      FROM merged WHERE ingredients IS NOT NULL GROUP BY 1 ORDER BY lists DESC""")

In [ ]:
# top categories
cats = q("SELECT category, count(*) AS products FROM merged WHERE category IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 20")
cats.set_index("category").products.sort_values().plot.barh(figsize=(8, 6))
plt.title("Top 20 categories"); plt.tight_layout(); plt.show()

## 6. Transform phase 4: enrich (recalls and confidence)

In [ ]:
t = time.time()
stats.update(transform.enrich(con))
print(f"{time.time() - t:.0f}s")
pd.Series({k: v for k, v in stats.items() if not isinstance(v, (dict, list))}, name="value").to_frame()

In [ ]:
q("""SELECT confidence, count(*) AS products, count(*) FILTER (WHERE sources & 4 > 0) AS with_recall
      FROM merged GROUP BY 1 ORDER BY 1 DESC""")

In [ ]:
# nutrient sanity check: kcal per 100 g against protein, carbs and fat (4/4/9 rule)
check = q("""SELECT kcal, 4 * protein_g + 4 * carbs_g + 9 * fat_g AS kcal_from_macros
               FROM merged WHERE kcal IS NOT NULL AND protein_g IS NOT NULL AND carbs_g IS NOT NULL AND fat_g IS NOT NULL
               USING SAMPLE 20000 ROWS (reservoir, 7)""")
check.plot.scatter("kcal_from_macros", "kcal", s=2, alpha=0.3, figsize=(5, 5))
plt.plot([0, 900], [0, 900], color="black", linewidth=1)
plt.title("kcal vs 4/4/9 from macros (sample of 20,000)"); plt.tight_layout(); plt.show()
print("off by more than 25%:", round(100 * ((check.kcal - check.kcal_from_macros).abs()
                                         > 0.25 * check.kcal_from_macros.clip(lower=20)).mean(), 1), "%")

## 7. Load: full database, phone catalog, server export
Writes `labelsage-full.sqlite`, `catalog.sqlite` (+ `.gz`) and `server/*.parquet` into `OUT`.

In [ ]:
t = time.time()
result = load.run(con, stats, OUT)
print(f"{time.time() - t:.0f}s")
pd.Series({k: v for k, v in result.items() if k != "full_rows"}, name="value").to_frame()

In [ ]:
pd.Series(result["full_rows"], name="rows").to_frame()

## 8. Input for the Amazon match step

In [ ]:
import sqlite3
with sqlite3.connect(OUT / "labelsage-full.sqlite") as db:
    mi = pd.read_sql("SELECT * FROM match_input LIMIT 10", db)
    n = db.execute("SELECT count(*) FROM match_input").fetchone()[0]
print(n, "products to match, most-scanned first")
mi

## 9. The model: identity check (OFF record vs FDC on the same barcode)
The matcher (`match/matcher.py`) is a rule-based scorer, not a trained model: brand and title similarity, minus a
penalty for differing critical words (flavors, allergens, "unsweetened"...), with vetoes and review caps.
Predicted match = tier `accept` = the OFF photo is kept. **Target: 0 false positives (precision 1.0)**, then as
much recall as possible.

Labels: `label-sample.csv` (round 1, stratified by confidence, split tune/holdout), `label-sample-round2.csv`
(200 accept / 50 review / 50 reject), `label-sample-round3.csv` (1,000 accepts). Labels come from a model
reading names blind to scores (`label_source = model`), not from people; rows noted `unsure:` need a human look.
Each pair is re-scored here with the matcher of this build, so older tiers in the CSVs don't matter.

In [ ]:
print("matcher", matcher.MATCHER_VERSION)
pd.DataFrame(matcher.THRESHOLDS, index=["accept_at", "review_at"]).T

In [ ]:
# synthetic pairs (hand-written edge cases, not real data)
rows = evaluate.report(ETL / "match" / "eval-pairs-synthetic-2026-10-28.json")

In [ ]:
# labelled real pairs, scored with this build's off_check tiers
tier_now = q("SELECT lpad(CAST(gtin AS VARCHAR), 14, '0') AS gtin, tier AS tier_now, confidence AS conf_now, reasons AS reasons_now FROM off_check")
def labelled(file, split=None):
    path = OUT / file
    if not path.exists():
        return pd.DataFrame()
    d = pd.read_csv(path, dtype={"gtin": str}).dropna(subset=["same_product"])
    d = d[d.split == split] if split else d
    return d.drop(columns=["tier", "confidence", "reasons"], errors="ignore").merge(tier_now, on="gtin")

samples = {"round 1 tune": labelled("label-sample.csv", "tune"), "round 1 holdout": labelled("label-sample.csv", "holdout"),
           "round 2 (all tiers)": labelled("label-sample-round2.csv"), "round 3 (accepts)": labelled("label-sample-round3.csv")}
def metrics(d):
    pred, actual = d.tier_now.eq("accept"), d.same_product.astype(int).eq(1)
    tp, fp, fn, tn = (pred & actual).sum(), (pred & ~actual).sum(), (~pred & actual).sum(), (~pred & ~actual).sum()
    return {"pairs": len(d), "accepted": int(pred.sum()), "false_pos": int(fp), "precision": round(tp / max(tp + fp, 1), 4),
            "recall_in_sample": round(tp / max(tp + fn, 1), 3), "missed_true": int(fn), "true_rejects": int(tn),
            "unsure_labels": int(d.label_note.fillna("").str.startswith("unsure").sum())}
pd.DataFrame({k: metrics(d) for k, d in samples.items() if len(d)}).T

In [ ]:
# how sure is "0 false positives"? one-sided 95% upper bound on the wrong-photo rate (Clopper-Pearson),
# over every accepted pair that has a label (rounds 2 and 3 are random samples of the accept tier)
from scipy.stats import beta
acc = pd.concat([samples["round 2 (all tiers)"], samples["round 3 (accepts)"]]).query("tier_now == 'accept'")
n, fp = len(acc), int((acc.same_product == 0).sum())
upper = beta.ppf(0.95, fp + 1, max(n - fp, 1))
accepted_total = q("SELECT count(*) AS n FROM off_check WHERE tier = 'accept'").n[0]
print(f"{n} labelled accepts, {fp} wrong -> wrong-photo rate at most {upper:.2%} (95%), "
      f"i.e. at most {upper * accepted_total:,.0f} of {accepted_total:,} accepted photos")

In [ ]:
# recall over the whole catalog (estimate): share of true matches per tier from round 2, times the tier sizes
r2 = samples["round 2 (all tiers)"]
sizes = q("SELECT tier, count(*) AS pairs FROM off_check GROUP BY tier").set_index("tier").pairs
share = r2.groupby("tier_now").same_product.mean()
est = pd.DataFrame({"pairs": sizes, "share_true_in_sample": share.round(3)})
est["est_true_matches"] = (est.pairs * est.share_true_in_sample).round()
print(f"estimated recall: {est.loc['accept', 'est_true_matches'] / est.est_true_matches.sum():.1%} "
      "of real OFF matches keep their photo")
est

In [ ]:
# what the review/reject tiers are made of (true matches we lose), from the labelled pairs
lost = pd.concat(samples.values()).query("tier_now != 'accept' and same_product == 1")
(lost.reasons_now.fillna("").str.split(",").explode().str.split(":").str[0].replace("", "(low similarity only)")
     .value_counts().rename("lost true matches").to_frame())

In [ ]:
# every labelled wrong pair and where the matcher put it (all should be review or reject)
wrong = pd.concat([d.assign(sample=k) for k, d in samples.items()]).query("same_product == 0")
wrong[["sample", "tier_now", "conf_now", "fdc_name", "off_name", "label_note", "reasons_now"]]

In [ ]:
# the allergen review cap (2026-10-28.4): one-sided allergen word and no brand to confirm it
cap = q("""SELECT regexp_extract(reasons, 'allergen_one_side:([a-z,]+)', 1) AS allergen, tier, count(*) AS pairs
             FROM off_check WHERE reasons LIKE '%allergen_one_side%' GROUP BY ALL ORDER BY pairs DESC""")
cap.pivot_table(index="allergen", columns="tier", values="pairs", fill_value=0).sort_values("review", ascending=False).head(12)

## 10. The phone catalog (`catalog.sqlite`)
What the app downloads every week: every recalled product plus the most-scanned products, under 60 MB.

In [ ]:
import sqlite3
cat = sqlite3.connect(OUT / "catalog.sqlite")
cq = lambda sql: pd.read_sql(sql, cat)
print(f"catalog.sqlite {(OUT / 'catalog.sqlite').stat().st_size / 1e6:.1f} MB, "
      f"download (.gz) {(OUT / 'catalog.sqlite.gz').stat().st_size / 1e6:.1f} MB")
cq("SELECT name AS part, round(sum(pgsize) / 1e6, 1) AS mb FROM dbstat GROUP BY name ORDER BY mb DESC LIMIT 8")

In [ ]:
# what a scan on the phone can show: coverage of each field among phone products
cq("""SELECT count(*) AS products,
          round(100.0 * avg(p.name IS NOT NULL), 1) AS pct_name,
          round(100.0 * avg(p.brand_id IS NOT NULL), 1) AS pct_brand,
          round(100.0 * avg(i.gtin IS NOT NULL), 1) AS pct_ingredients,
          round(100.0 * avg(n.gtin IS NOT NULL), 1) AS pct_nutrients,
          round(100.0 * avg(p.image IS NOT NULL), 1) AS pct_photo,
          round(100.0 * avg(p.flags & 1 > 0), 1) AS pct_recall_stub,
          round(100.0 * avg(p.flags & 8 > 0), 1) AS pct_recipe_changed
       FROM product p LEFT JOIN ingredients i USING (gtin) LEFT JOIN nutrient n USING (gtin)""").T

In [ ]:
# verdict readiness: a product can get a safe / not-safe verdict only with an ingredient list;
# without one the app says "can't confirm"
ready = cq("""SELECT CASE WHEN p.flags & 1 > 0 THEN 'recall stub (no data)'
                          WHEN i.gtin IS NULL THEN "no ingredients: can't confirm"
                          WHEN p.image IS NULL THEN 'verdict, no photo' ELSE 'verdict + photo' END AS scan_result,
                     count(*) AS products
              FROM product p LEFT JOIN ingredients i USING (gtin) GROUP BY 1 ORDER BY 2 DESC""")
ready.set_index("scan_result").products.plot.barh(figsize=(8, 2.8)); plt.title("What a scan shows on the phone")
plt.tight_layout(); plt.show(); ready

In [ ]:
cq("""SELECT CASE confidence WHEN 2 THEN 'high' WHEN 1 THEN 'medium' ELSE 'low' END AS confidence,
          CASE WHEN image LIKE 'http%' THEN 'Amazon' WHEN image IS NOT NULL THEN 'OFF' ELSE 'none' END AS photo,
          count(*) AS products
       FROM product GROUP BY 1, 2 ORDER BY products DESC""")

In [ ]:
top_brands = cq("""SELECT b.name AS brand, count(*) AS products, round(100.0 * avg(p.image IS NOT NULL), 0) AS pct_photo
                   FROM product p JOIN brand b USING (brand_id) GROUP BY 1 ORDER BY 2 DESC LIMIT 15""")
top_brands

In [ ]:
cq("""SELECT c.name AS category, count(*) AS products FROM product p JOIN category c USING (category_id)
       GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").set_index("category").products.sort_values().plot.barh(figsize=(8, 5))
plt.title("Top categories on the phone"); plt.tight_layout(); plt.show()

## 11. The server export (`server/*.parquet`)
The whole catalog for lookups the phone can't answer, plus the audit tables (field sources, merge conflicts).

In [ ]:
manifest = json.loads((OUT / "server" / "manifest.json").read_text())
pd.DataFrame(manifest).T.assign(mb=lambda d: (d.bytes.astype(float) / 1e6).round(1))[["file", "rows", "mb"]].sort_values("rows", ascending=False)

In [ ]:
prod = str(OUT / "server" / "product.parquet")
q(f"""SELECT CASE WHEN in_fdc = 1 AND in_off = 1 THEN 'FDC + OFF' WHEN in_fdc = 1 THEN 'FDC only'
                   WHEN is_recall_stub = 1 THEN 'recall stub' ELSE 'OFF only' END AS sources,
              count(*) AS products, sum(on_phone) AS on_phone,
              count(*) FILTER (WHERE image_url IS NOT NULL) AS with_photo,
              count(*) FILTER (WHERE off_match_tier = 'accept') AS off_accepted,
              count(*) FILTER (WHERE off_match_tier = 'review') AS off_review,
              count(*) FILTER (WHERE off_match_tier = 'reject') AS off_rejected
       FROM '{prod}' GROUP BY 1 ORDER BY products DESC""")

In [ ]:
# where each product's photo comes from, and how the photo rate tracks popularity (OFF scans)
q(f"""SELECT CASE WHEN unique_scans IS NULL OR unique_scans = 0 THEN '0 scans' WHEN unique_scans < 10 THEN '1-9'
                   WHEN unique_scans < 100 THEN '10-99' ELSE '100+' END AS off_scans,
              count(*) AS products, round(100.0 * avg(CAST(image_url IS NOT NULL AS INTEGER)), 1) AS pct_photo,
              round(100.0 * avg(on_phone), 1) AS pct_on_phone
       FROM '{prod}' GROUP BY 1 ORDER BY min(coalesce(unique_scans, 0))""")

In [ ]:
# which source won each field (audit table: one row per product and field)
pfs = str(OUT / "server" / "product_field_source.parquet")
q(f"SELECT field, source, count(*) AS products FROM '{pfs}' GROUP BY ALL") \
    .pivot_table(index="field", columns="source", values="products", fill_value=0)

## 12. Summary: the numbers to remember from this build

In [ ]:
summary = {
    "products (full catalog)": result["full_rows"]["product"],
    "products on the phone": result["phone_products"],
    "phone download MB": result["download_mb"],
    "FDC+OFF pairs checked": int(sizes.sum()),
    "OFF photos accepted": int(sizes.get("accept", 0)),
    "matcher version": matcher.MATCHER_VERSION,
    "labelled accepts / wrong": f"{n} / {fp}",
    "wrong-photo rate, 95% upper": f"{upper:.2%}",
    "estimated photo recall": f"{est.loc['accept', 'est_true_matches'] / est.est_true_matches.sum():.1%}",
    "phone products with verdict data": f"{100 * ready.loc[ready.scan_result.str.startswith('verdict'), 'products'].sum() / ready.products.sum():.1f}%",
}
pd.Series(summary, name="value").to_frame()

In [ ]:
cat.close()
con.close()